# Session 4 — Python for Market Risk
## Downloading Stock Data, Return Distributions & Portfolio Risk (VaR / CVaR)

**MSc Finance | Risk Modelling Course**

---

### Learning Objectives
By the end of this session you will be able to:
1. Download and clean historical price data using `yfinance`
2. Compute log-returns and understand their statistical properties
3. Visualise return distributions and rolling volatility with `seaborn`
4. Build a simple multi-asset portfolio and compute its risk metrics
5. Calculate **Historical VaR** and **CVaR** (Expected Shortfall) at multiple confidence levels
6. Appreciate how volatility regimes change over time (pre/post COVID case)

---

### Style note — method chaining
Throughout this notebook we favour **method chaining** (inspired by R's `dplyr` pipes `%>%`).  
In pandas this means stringing `.method()` calls together rather than creating lots of
intermediate variables.  The pattern looks like:

```python
result = (
    df
    .pipe(some_function)        # custom function injection
    .assign(new_col=...)        # add / transform columns
    .query("condition")         # filter rows
    .groupby("key")             # aggregate
    .agg({"col": "mean"})
    .rename(columns={"col": "nice_name"})
    .reset_index()
)
```

This keeps code readable and makes the data transformation story easy to follow.


## 0. Imports & Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

# ── Data & numerics ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import yfinance as yf

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ── Seaborn global theme ──────────────────────────────────────────────────────
# 'whitegrid' gives clean axes; palette 'tab10' is colourblind-friendly
sns.set_theme(style="whitegrid", palette="tab10", font_scale=1.15)

# ── Display options ───────────────────────────────────────────────────────────
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 20)

print("All libraries loaded successfully.")


## 1. Downloading Historical Price Data

We pull **adjusted closing prices** for five assets that span different sectors,
giving us a nicely diversified illustrative portfolio:

| Ticker | Company / Index | Sector |
|--------|----------------|--------|
| AAPL   | Apple Inc.      | Technology |
| JPM    | JPMorgan Chase  | Financials |
| XOM    | ExxonMobil      | Energy |
| KO     | Coca-Cola       | Consumer Staples |
| ^GSPC  | S&P 500 Index   | Broad Market Benchmark |

We deliberately span **2018–2024** to capture normal markets, the COVID crash (Feb–Mar 2020),
and the post-pandemic inflation / rate-rise period.


In [ ]:
# ── Define our universe ───────────────────────────────────────────────────────
TICKERS    = ["AAPL", "JPM", "XOM", "KO", "^GSPC"]
START_DATE = "2018-01-01"
END_DATE   = "2024-12-31"

# ── Download via yfinance ──────────────────────────────────────────────────────
# auto_adjust=True gives dividend / split adjusted prices automatically
raw = yf.download(TICKERS, start=START_DATE, end=END_DATE,
                  auto_adjust=True, progress=False)

# yfinance returns a MultiIndex column like (field, ticker).
# We only need 'Close' prices — select and clean in one chain.
prices = (
    raw["Close"]                          # select the 'Close' level
    .rename_axis("Date")                  # make index name explicit
    .rename_axis("Ticker", axis="columns")# make column axis name explicit
    .dropna(how="all")                    # drop rows where ALL prices are NaN
    .ffill()                              # forward-fill any remaining gaps (holidays)
)

print(f"Price matrix shape: {prices.shape}  ({prices.shape[0]} trading days × {prices.shape[1]} assets)")
prices.tail()


## 2. Computing Log-Returns

We use **log-returns** rather than simple returns because:
- They are time-additive: a multi-period log-return is the sum of single-period ones
- They are approximately normally distributed for short horizons
- They eliminate the lower bound at −100% that afflicts simple returns

$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)$$


In [ ]:
# ── Log-return computation via method chain ───────────────────────────────────
# np.log(prices / prices.shift(1)) is equivalent to prices.pct_change() on log scale
returns = (
    np.log(prices / prices.shift(1))     # compute daily log-returns
    .dropna()                             # drop the first row (NaN from shift)
)

print(f"Returns matrix shape: {returns.shape}")
returns.describe().T


## 3. Summary Statistics — Annualised Risk & Return

We annualise by multiplying mean returns by **252** (trading days per year)
and standard deviation by **√252**.

Note: The S&P 500 (`^GSPC`) serves as our benchmark throughout.


In [ ]:
TRADING_DAYS = 252

def annualised_stats(returns_df):
    """
    Given a DataFrame of daily log-returns, return a summary table
    with annualised mean return, volatility, Sharpe-like ratio, skew and kurtosis.
    Risk-free rate assumed 0 for simplicity (adjust RF below if needed).
    """
    RF = 0.04 / TRADING_DAYS  # ~4% annualised risk-free rate, daily

    stats = (
        returns_df
        .agg(["mean", "std", "skew", "kurt"])     # four moments
        .T                                          # transpose so assets are rows
        .rename(columns={"mean": "Daily Mean",
                         "std":  "Daily Std",
                         "skew": "Skewness",
                         "kurt": "Excess Kurtosis"})
        .assign(
            Ann_Return = lambda df: df["Daily Mean"] * TRADING_DAYS,
            Ann_Vol    = lambda df: df["Daily Std"]  * np.sqrt(TRADING_DAYS),
        )
        .assign(
            Sharpe = lambda df: (df["Ann_Return"] - 0.04) / df["Ann_Vol"]
        )
        [["Ann_Return", "Ann_Vol", "Sharpe", "Skewness", "Excess Kurtosis"]]
    )
    return stats

stats = annualised_stats(returns)
stats


## 4. Visualising Return Distributions

A normal distribution assumption is baked into many risk models (including Black-Scholes).
Let's see how realistic that is — financial returns are typically **fat-tailed** (leptokurtic)
and often slightly **negatively skewed** (large losses are more common than a normal predicts).


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, ticker in enumerate(TICKERS):
    ax = axes[i]
    data = returns[ticker].dropna()

    # ── Seaborn histplot with KDE overlay ─────────────────────────────────────
    sns.histplot(data, bins=80, kde=True, stat="density",
                 color=sns.color_palette("tab10")[i], alpha=0.55, ax=ax)

    # ── Overlay a fitted normal for comparison ─────────────────────────────────
    from scipy.stats import norm
    mu, sigma = data.mean(), data.std()
    x = np.linspace(data.min(), data.max(), 300)
    ax.plot(x, norm.pdf(x, mu, sigma), "k--", lw=1.5, label="Normal fit")

    # ── Annotation ────────────────────────────────────────────────────────────
    skew_val = data.skew()
    kurt_val = data.kurt()
    ax.set_title(f"{ticker}", fontsize=13, fontweight="bold")
    ax.set_xlabel("Daily Log-Return")
    ax.text(0.03, 0.95, f"Skew={skew_val:.2f}\nKurt={kurt_val:.2f}",
            transform=ax.transAxes, va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))
    ax.legend(fontsize=8)

# ── Remove the unused 6th subplot ─────────────────────────────────────────────
axes[5].set_visible(False)

fig.suptitle("Daily Log-Return Distributions vs Normal Fit\n(2018–2024)",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

print("\nKey insight: The KDE (solid) has HIGHER peaks and FATTER tails than the normal (dashed).")
print("This is 'excess kurtosis' — large moves happen more often than a normal distribution predicts.")


## 5. Rolling Volatility — Spotting Regime Changes

**Static** volatility (one number for the whole period) ignores the fact that markets
are calm for extended stretches and then violently volatile for short bursts.
Rolling volatility makes this visible.

We use a **21-day** rolling window (≈ 1 trading month) and annualise.


In [ ]:
ROLL_WINDOW = 21  # trading days ≈ 1 month

rolling_vol = (
    returns
    .rolling(window=ROLL_WINDOW)          # rolling window object
    .std()                                 # rolling std dev
    .mul(np.sqrt(TRADING_DAYS))           # annualise
    .dropna()
    .rename_axis("Date")
)

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

for ticker in TICKERS:
    ax.plot(rolling_vol.index, rolling_vol[ticker], lw=1.3, label=ticker)

# ── Annotate key events ────────────────────────────────────────────────────────
events = {
    "COVID crash\n(Mar 2020)":  "2020-03-16",
    "Fed hikes begin\n(Mar 2022)": "2022-03-16",
}
for label, date in events.items():
    ax.axvline(pd.Timestamp(date), color="red", lw=1.2, ls="--", alpha=0.7)
    ax.text(pd.Timestamp(date), ax.get_ylim()[1] * 0.92, label,
            ha="center", fontsize=8.5, color="red",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.6))

ax.set_title(f"{ROLL_WINDOW}-Day Rolling Annualised Volatility", fontsize=14, fontweight="bold")
ax.set_ylabel("Annualised Volatility")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

print("\nKey insight: Volatility CLUSTERS — it is persistently high or low, not random.")
print("The COVID spike in March 2020 is clearly visible across all assets.")


## 6. Correlation Matrix

Correlation is the engine of diversification.
A portfolio's risk is **not** the average of individual risks — correlations below 1
reduce total portfolio volatility.

We compute correlations in two regimes to show how they shift under stress.


In [ ]:
# ── Define two market regimes ─────────────────────────────────────────────────
regimes = {
    "Normal (2018–2019)": ("2018-01-01", "2019-12-31"),
    "Stress (COVID 2020)": ("2020-01-01", "2020-12-31"),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, (start, end)) in zip(axes, regimes.items()):
    corr = (
        returns
        .loc[start:end]          # filter to regime dates
        .corr()                  # pairwise Pearson correlation
    )
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # upper triangle mask

    sns.heatmap(
        corr, ax=ax,
        annot=True, fmt=".2f",
        cmap="RdYlGn", center=0, vmin=-1, vmax=1,
        mask=mask,                # show lower triangle only
        linewidths=0.5,
        cbar_kws={"shrink": 0.8},
    )
    ax.set_title(f"Correlation Matrix\n{label}", fontsize=12, fontweight="bold")

plt.suptitle("Correlations Rise During Stress — Diversification Fails When You Need It Most",
             fontsize=12, style="italic", y=1.02)
plt.tight_layout()
plt.show()


## 7. Building a Simple Equal-Weight Portfolio

We allocate capital equally across the four stocks (excluding the index benchmark).
This is the simplest possible diversification strategy — and often surprisingly hard to beat.

Portfolio daily return: $r_p = \sum_i w_i \cdot r_i$ where $w_i = 0.25$ for each stock.


In [ ]:
# ── Define portfolio (equal-weight, 4 stocks, no index) ───────────────────────
PORTFOLIO_TICKERS = ["AAPL", "JPM", "XOM", "KO"]
N_ASSETS = len(PORTFOLIO_TICKERS)
weights  = np.array([1 / N_ASSETS] * N_ASSETS)  # equal weights: 0.25 each

port_returns = (
    returns[PORTFOLIO_TICKERS]                # select only portfolio assets
    .assign(Portfolio=lambda df: df.dot(weights))  # weighted sum → portfolio return
)

print("Daily portfolio return statistics:")
print(port_returns["Portfolio"].describe())
print(f"\nAnnualised Return : {port_returns['Portfolio'].mean() * TRADING_DAYS:.2%}")
print(f"Annualised Volatility: {port_returns['Portfolio'].std() * np.sqrt(TRADING_DAYS):.2%}")


## 8. Historical Value at Risk (VaR) and CVaR (Expected Shortfall)

### What is VaR?
**Value at Risk** at confidence level $\alpha$ answers:
> *"What is the worst daily loss I can expect to suffer on $X\%$ of trading days?"*

Formally: $\text{VaR}_{\alpha} = -\text{quantile}_{1-\alpha}(r)$

At 95% confidence: on 95% of days, losses will NOT exceed VaR₉₅.
Equivalently, VaR is exceeded on 5% of days — roughly 13 trading days per year.

### What is CVaR / Expected Shortfall?
CVaR fixes VaR's main weakness: VaR says nothing about *how bad* the bad days are.

**CVaR** = average loss **given** that we are already beyond the VaR threshold:
$$\text{CVaR}_{\alpha} = -\mathbb{E}[r \mid r < -\text{VaR}_{\alpha}]$$

CVaR is **coherent** (sub-additive), making it preferred by regulators (Basel III uses CVaR).


In [ ]:
def compute_var_cvar(returns_series, confidence_levels=(0.90, 0.95, 0.99)):
    """
    Compute Historical (non-parametric) VaR and CVaR for a returns series.

    Parameters
    ----------
    returns_series   : pd.Series of daily log-returns
    confidence_levels: tuple of floats, e.g. (0.95, 0.99)

    Returns
    -------
    pd.DataFrame with VaR and CVaR at each confidence level (as positive loss figures)
    """
    rows = []
    for cl in confidence_levels:
        var   = -returns_series.quantile(1 - cl)           # negate → positive loss
        cvar  = -returns_series[returns_series < -var].mean()  # average of tail losses
        rows.append({"Confidence": f"{cl:.0%}", "VaR": var, "CVaR": cvar})

    return pd.DataFrame(rows).set_index("Confidence")


# ── Compute for each asset + portfolio ────────────────────────────────────────
all_series = port_returns.copy()  # includes individual stocks + portfolio column

results = {}
for col in all_series.columns:
    results[col] = compute_var_cvar(all_series[col])

# ── Display for the portfolio ──────────────────────────────────────────────────
print("=== Portfolio VaR & CVaR ===")
print(results["Portfolio"])
print()

# ── Cross-asset comparison at 95% ─────────────────────────────────────────────
comparison_95 = (
    pd.DataFrame({
        ticker: results[ticker].loc["95%"]
        for ticker in all_series.columns
    })
    .T
    .assign(
        Ann_VaR  = lambda df: df["VaR"]  * np.sqrt(TRADING_DAYS),
        Ann_CVaR = lambda df: df["CVaR"] * np.sqrt(TRADING_DAYS),
    )
    .rename(columns={"VaR": "Daily VaR", "CVaR": "Daily CVaR"})
)

print("=== Cross-Asset Comparison at 95% Confidence ===")
comparison_95


In [ ]:
# ── Visualise VaR on the return distribution ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, ticker in zip(axes, ["^GSPC", "Portfolio"]):
    data   = all_series[ticker].dropna()
    var_95 = results[ticker].loc["95%", "VaR"]
    var_99 = results[ticker].loc["99%", "VaR"]

    # Full distribution
    sns.histplot(data, bins=80, kde=True, stat="density",
                 color="steelblue", alpha=0.4, ax=ax, label="Return dist.")

    # Shade the tail losses
    tail_data = data[data < -var_99]
    ax.fill_between(
        sorted(tail_data), 0,
        [sns.kdeplot(data, ax=None).get_lines()[0].get_ydata().max() * 0.1] * len(tail_data),
        color="darkred", alpha=0.5, label="CVaR region (99%)"
    )

    # VaR vertical lines
    ax.axvline(-var_95, color="orange", lw=2, ls="--", label=f"VaR 95% = {var_95:.2%}")
    ax.axvline(-var_99, color="red",    lw=2, ls="--", label=f"VaR 99% = {var_99:.2%}")

    ax.set_title(f"{ticker} — Return Distribution with VaR", fontweight="bold")
    ax.set_xlabel("Daily Log-Return")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


## 9. Regime Analysis — Pre vs Post COVID Volatility

This is a concrete example of why **static volatility assumptions are dangerous**.
Models calibrated on 2018–2019 data would dramatically underestimate risk during 2020.


In [ ]:
# ── Define regimes ────────────────────────────────────────────────────────────
regime_map = {
    "Pre-COVID (2018–2019)": ("2018-01-01", "2019-12-31"),
    "COVID Crash (2020)":    ("2020-01-01", "2020-12-31"),
    "Recovery (2021–2022)":  ("2021-01-01", "2022-12-31"),
}

regime_stats = (
    pd.concat(
        {
            label: annualised_stats(returns[PORTFOLIO_TICKERS].loc[start:end])
            for label, (start, end) in regime_map.items()
        },
        names=["Regime", "Ticker"]
    )
    .reset_index()
)

# ── Plot annualised volatility by regime ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

sns.barplot(
    data=regime_stats,
    x="Ticker", y="Ann_Vol",
    hue="Regime",
    palette=["#4CAF50", "#F44336", "#2196F3"],
    ax=ax
)

ax.set_title("Annualised Volatility by Market Regime", fontsize=14, fontweight="bold")
ax.set_ylabel("Annualised Volatility")
ax.set_xlabel("")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend(title="Regime", loc="upper left")
plt.tight_layout()
plt.show()

print("\nKey insight: COVID-era volatility was 2–4× the pre-COVID baseline.")
print("A risk model using pre-COVID vol would have massively under-reserved capital.")


## 10. Cumulative Returns — Growth of £1 Invested

Let's close with a practical view: what would £1 invested at the start of 2018 be worth by end 2024?


In [ ]:
# ── Cumulative log-return → price index ───────────────────────────────────────
# exp(cumulative log-returns) restores the compound growth factor
cum_returns = (
    port_returns
    .cumsum()                   # cumulative sum of log-returns
    .apply(np.exp)              # convert log-cumulative → simple growth factor
    .rename(columns={"Portfolio": "Portfolio (EW)"})
)

fig, ax = plt.subplots(figsize=(13, 5))

for col in cum_returns.columns:
    lw = 2.5 if col == "Portfolio (EW)" else 1.2
    ls = "-"  if col == "Portfolio (EW)" else "--"
    ax.plot(cum_returns.index, cum_returns[col], lw=lw, ls=ls, label=col)

ax.axhline(1, color="grey", lw=0.8, ls=":")
ax.set_title("Cumulative Growth of £1 Invested (2018–2024)", fontsize=14, fontweight="bold")
ax.set_ylabel("Portfolio Value (£ per £1 invested)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend()
plt.tight_layout()
plt.show()


## Summary & Key Takeaways

| Concept | What we learned |
|---------|----------------|
| Log-returns | Time-additive, approximately normal but with fat tails in practice |
| Rolling volatility | Volatility clusters — it is NOT constant (this will matter for Black-Scholes) |
| Correlations | Rise dramatically during stress — diversification weakens precisely when needed |
| Historical VaR | Simple but relies entirely on the past; says nothing about tail shape |
| CVaR | Tells us the *average* loss in the tail — preferred by regulators (Basel III) |
| Regime analysis | Static models calibrated on calm periods dramatically understate crisis risk |

### Looking ahead to Session 5
We'll take the **GBM / lognormal** model of stock prices seriously and use it to price options
analytically (Black-Scholes formula) and by simulation — and then examine where it breaks down.


---
---

# 🧪 Portfolio Exercises — Connecting to a Real Portfolio Viewer App

The following exercises are based on a real-world **Dash application** (`app.py`) that
visualises a multi-currency equity portfolio with:
- Performance & CAPM analytics (Tab 1)
- FX-decomposed regional returns (Tab 2)
- An interactive date-range slider that filters all charts live

In these exercises you will **recreate the core analytical engine** of that app — the
pandas / numpy logic that sits behind every chart and table — as pure notebook code.

This is deliberate: understanding the data pipeline in a notebook makes you a far better
developer when you later move to a production app framework.

---

## App Architecture Overview

```
portfolio.csv   ←  weights, names, tickers
prices.csv      ←  downloaded from Yahoo Finance (yfinance)
app.py          ←  Dash app: date slider → filter prices → build_tab1() / build_tab2()
                                              ↓
                            charts & tables returned to browser
```

The key insight is that `build_tab1()` and `build_tab2()` are **pure functions**:
they take a filtered price DataFrame and return figures + table data.
Below you will implement each step that those functions perform.

---

## Portfolio Configuration (from `app.py`)

The app defines three dictionaries you will need throughout:
- `TICKER_MAP`   — maps internal portfolio tickers → Yahoo Finance tickers
- `CURRENCY_MAP` — maps portfolio tickers → denomination currency
- `FX_TICKERS`   — maps currency codes → Yahoo Finance FX pair tickers


## Exercise Setup — Paste in the App's Configuration

In [ ]:

# ── Re-use the same imports from earlier in the session ───────────────────────
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="tab10", font_scale=1.15)
pd.set_option("display.float_format", "{:.4f}".format)

# ── Ticker and currency configuration (directly from app.py) ──────────────────
TICKER_MAP = {
    "MSFT":"MSFT", "GOOG":"GOOG", "V":"V",    "MA":"MA",
    "SCHW":"SCHW", "TSM":"TSM",   "ASML":"ASML","INTU":"INTU",
    "GE":"GE",     "EW":"EW",     "ZTS":"ZTS", "FERG":"FERG",
    "ALLE":"ALLE", "BKNG":"BKNG", "WDAY":"WDAY","AZO":"AZO",
    "ADSK":"ADSK", "EFX":"EFX",
    "B0SWJX":"LSEG.L","BVZK7T":"ULVR.L","B19NLV":"EXPN.L","B082RF":"RTO.L",
    "702196":"DB1.DE","B058TZ":"SAF.PA","588185":"EVD.DE",
    "711038":"ROG.SW","B4TX8S":"1299.HK","BK1N46":"HDFCBANK.NS",
    "BLDBN5":"ATCO-B.ST","BG36ZK":"B3SA3.SA","B01C1P":"BBCA.JK",
}

CURRENCY_MAP = {
    "MSFT":"USD","GOOG":"USD","V":"USD","MA":"USD","SCHW":"USD","TSM":"USD",
    "ASML":"USD","INTU":"USD","GE":"USD","EW":"USD","ZTS":"USD","FERG":"USD",
    "ALLE":"USD","BKNG":"USD","WDAY":"USD","AZO":"USD","ADSK":"USD","EFX":"USD",
    "B0SWJX":"GBP","BVZK7T":"GBP","B19NLV":"GBP","B082RF":"GBP",
    "702196":"EUR","B058TZ":"EUR","588185":"EUR",
    "711038":"CHF","B4TX8S":"HKD","BK1N46":"INR",
    "BLDBN5":"SEK","BG36ZK":"BRL","B01C1P":"IDR",
}

FX_TICKERS = {
    "GBP":"GBPUSD=X","EUR":"EURUSD=X","CHF":"CHFUSD=X",
    "HKD":"HKDUSD=X","INR":"INRUSD=X","SEK":"SEKUSD=X",
    "BRL":"BRLUSD=X","IDR":"IDRUSD=X",
}

# UK stocks are priced in pence on Yahoo — divide by 100 to get GBP
GBP_PENCE_TICKERS = {"LSEG.L", "ULVR.L", "EXPN.L", "RTO.L"}

REGION_LABEL = {
    "USD":"North America (USD)","GBP":"UK (GBP)","EUR":"Europe (EUR)",
    "CHF":"Switzerland (CHF)","HKD":"Hong Kong (HKD)","INR":"India (INR)",
    "SEK":"Sweden (SEK)","BRL":"Brazil (BRL)","IDR":"Indonesia (IDR)",
}

TRADING_DAYS = 252
START_DATE   = "2021-01-01"

print("Configuration loaded.")
print(f"Portfolio tickers : {len(TICKER_MAP)}")
print(f"Currencies tracked: {len(FX_TICKERS)}")


---
## Exercise 1 — Download and Assemble the Price Matrix

### Background
The app's `download_prices()` function pulls adjusted closing prices for:
1. All Yahoo Finance tickers in `TICKER_MAP`
2. The S&P 500 benchmark (`^GSPC`)
3. All FX pairs in `FX_TICKERS`

It stores everything in a single wide DataFrame (`prices.csv`) where each column
is one ticker and each row is one trading day.

### Your Task
Write a function `download_all_prices(start, end)` that:
1. Builds the full list of tickers (equities + benchmark + FX)
2. Downloads adjusted close prices using `yfinance`
3. Forward-fills any gaps (public holidays differ across exchanges)
4. Returns a clean DataFrame

Then download prices from `2021-01-01` to today and inspect the result.

> **Hint:** Use `yf.download(..., auto_adjust=True)` and select `["Close"]` from the result.
> Some tickers may have missing columns — don't worry, `usd_price_series()` handles this later.


In [ ]:

# ── SOLUTION ──────────────────────────────────────────────────────────────────

def download_all_prices(start: str, end: str) -> pd.DataFrame:
    """
    Download adjusted closing prices for the full portfolio universe.

    Combines equity tickers, the S&P 500 benchmark, and all FX pairs
    into a single wide DataFrame.  Missing values are forward-filled
    (different exchanges observe different public holidays).

    Parameters
    ----------
    start : ISO date string e.g. '2021-01-01'
    end   : ISO date string e.g. '2024-12-31'

    Returns
    -------
    pd.DataFrame  — index=Date, columns=Yahoo tickers
    """
    # ── Build the full ticker list ─────────────────────────────────────────────
    equity_tickers = sorted(set(TICKER_MAP.values()))
    benchmark      = ["^GSPC"]
    fx_tickers     = list(FX_TICKERS.values())
    all_tickers    = equity_tickers + benchmark + fx_tickers

    print(f"Downloading {len(all_tickers)} tickers ({start} → {end}) …")

    # ── Download via yfinance ─────────────────────────────────────────────────
    raw = yf.download(all_tickers, start=start, end=end,
                      auto_adjust=True, progress=False)

    # yfinance returns a MultiIndex (field × ticker) when multiple tickers passed
    prices = (
        raw["Close"]            # select Close prices only
        .rename_axis("Date")    # name the index
        .rename_axis("Ticker", axis="columns")
        .ffill()                # forward-fill gaps (holidays, weekends already excluded)
    )

    print(f"Download complete.  Shape: {prices.shape}")
    print(f"Date range: {prices.index[0].date()} → {prices.index[-1].date()}")
    return prices


# ── Download data ──────────────────────────────────────────────────────────────
END_DATE = pd.Timestamp.today().strftime("%Y-%m-%d")
prices = download_all_prices(START_DATE, END_DATE)

# ── Quick inspection using method chaining ─────────────────────────────────────
# How many tickers have < 20% data coverage? (might indicate delisted / new listings)
coverage = (
    prices
    .notna()                                    # boolean mask
    .mean()                                     # fraction of non-null rows per column
    .rename("coverage")
    .to_frame()
    .assign(pct=lambda df: df["coverage"] * 100)
    .query("pct < 80")                          # flag sparse tickers
    .sort_values("pct")
)

print(f"\nTickers with <80% data coverage ({len(coverage)} found):")
print(coverage.to_string() if not coverage.empty else "None — full coverage!")
prices.tail(3)


---
## Exercise 2 — Build the Holdings Table from a CSV

### Background
The app reads `portfolio.csv` which has columns: `Ticker`, `name`, `weight`.
It filters out cash rows (tickers starting with `CASH_`) and zero-weight rows,
then normalises weights to sum to 1.

### Your Task
Replicate this using pandas method chaining.

Since you may not have `portfolio.csv`, we'll construct a representative
holdings table directly — this mirrors exactly what the app does after reading the CSV.

> **Key concept:** In the app, `holdings["w"]` is the normalised weight used
> for all portfolio return calculations.


In [ ]:

# ── SOLUTION ──────────────────────────────────────────────────────────────────

# ── Construct a representative holdings table ─────────────────────────────────
# In production this comes from portfolio.csv; here we build it manually
# using the same tickers and approximate weights as a real global equity fund

raw_holdings = pd.DataFrame([
    # USD stocks
    {"Ticker":"MSFT",   "name":"Microsoft",          "weight":6.5},
    {"Ticker":"GOOG",   "name":"Alphabet",           "weight":5.8},
    {"Ticker":"V",      "name":"Visa",               "weight":4.2},
    {"Ticker":"MA",     "name":"Mastercard",         "weight":4.0},
    {"Ticker":"SCHW",   "name":"Charles Schwab",     "weight":3.1},
    {"Ticker":"TSM",    "name":"TSMC (ADR)",         "weight":3.5},
    {"Ticker":"ASML",   "name":"ASML (USD-listed)",  "weight":3.0},
    {"Ticker":"INTU",   "name":"Intuit",             "weight":2.8},
    {"Ticker":"GE",     "name":"GE Aerospace",       "weight":2.5},
    {"Ticker":"EW",     "name":"Edwards Lifesciences","weight":2.2},
    {"Ticker":"ZTS",    "name":"Zoetis",             "weight":2.0},
    {"Ticker":"FERG",   "name":"Ferguson",           "weight":1.8},
    {"Ticker":"ALLE",   "name":"Allegion",           "weight":1.5},
    {"Ticker":"BKNG",   "name":"Booking Holdings",  "weight":3.2},
    {"Ticker":"WDAY",   "name":"Workday",            "weight":2.1},
    {"Ticker":"AZO",    "name":"AutoZone",           "weight":2.4},
    {"Ticker":"ADSK",   "name":"Autodesk",           "weight":1.9},
    {"Ticker":"EFX",    "name":"Equifax",            "weight":1.7},
    # GBP stocks
    {"Ticker":"B0SWJX", "name":"LSEG",              "weight":3.8},
    {"Ticker":"BVZK7T", "name":"Unilever",          "weight":2.9},
    {"Ticker":"B19NLV", "name":"Experian",          "weight":2.7},
    {"Ticker":"B082RF", "name":"Rentokil",          "weight":1.6},
    # EUR stocks
    {"Ticker":"702196", "name":"Deutsche Börse",    "weight":2.3},
    {"Ticker":"B058TZ", "name":"Safran",            "weight":2.1},
    {"Ticker":"588185", "name":"Evotec",            "weight":0.8},
    # Other regions
    {"Ticker":"711038", "name":"Roche",             "weight":3.1},
    {"Ticker":"B4TX8S", "name":"AIA Group",         "weight":1.9},
    {"Ticker":"BK1N46", "name":"HDFC Bank",         "weight":1.4},
    {"Ticker":"BLDBN5", "name":"Atlas Copco",       "weight":1.5},
    {"Ticker":"BG36ZK", "name":"B3 (Brazil)",       "weight":0.9},
    {"Ticker":"B01C1P", "name":"BCA (Indonesia)",   "weight":0.7},
    # Cash (should be filtered out)
    {"Ticker":"CASH_USD", "name":"USD Cash",        "weight":1.2},
])

# ── Replicate the app's holdings construction via method chaining ─────────────
holdings = (
    raw_holdings
    # Filter: no cash rows, no zero weights, must be in TICKER_MAP
    .query("not Ticker.str.startswith('CASH_')")
    .query("weight > 0")
    .loc[lambda df: df["Ticker"].isin(TICKER_MAP)]
    # Normalise weights to sum to 1
    .assign(w=lambda df: df["weight"] / df["weight"].sum())
    # Add derived columns useful for analysis
    .assign(
        currency = lambda df: df["Ticker"].map(CURRENCY_MAP),
        region   = lambda df: df["Ticker"].map(CURRENCY_MAP).map(REGION_LABEL),
        yf_ticker= lambda df: df["Ticker"].map(TICKER_MAP),
    )
    .reset_index(drop=True)
)

print(f"Holdings loaded: {len(holdings)} positions")
print(f"Weights sum to: {holdings['w'].sum():.6f}  (should be 1.0)")
print()

# ── Weight by region ──────────────────────────────────────────────────────────
region_weights = (
    holdings
    .groupby("region")["w"]
    .sum()
    .mul(100)
    .rename("Weight (%)")
    .sort_values(ascending=False)
    .round(1)
)
print("Portfolio weight by region:")
print(region_weights.to_string())
holdings[["Ticker","name","weight","w","currency","region"]].head(10)


---
## Exercise 3 — USD Price Conversion & Portfolio Return Calculation

### Background
The app's `usd_price_series()` function converts each holding's local price to USD:
1. For USD stocks: no conversion needed
2. For GBP stocks: divide by 100 (pence → GBP), then multiply by GBPUSD rate
3. For all others: multiply local price by the relevant FX rate

Once all prices are in USD, the app computes daily returns and calculates a
**weighted portfolio return**: $r_p = \sum_i w_i \cdot r_i$

### Your Task
1. Implement `usd_price_series()` as a clean pandas function
2. Build the full USD return matrix for all holdings
3. Compute the weighted portfolio return series
4. Plot it against the S&P 500 as a cumulative return chart using seaborn/matplotlib


In [ ]:

# ── SOLUTION ──────────────────────────────────────────────────────────────────

def usd_price_series(prices: pd.DataFrame, port_ticker: str) -> pd.Series | None:
    """
    Convert a holding's local price series to USD.

    Logic (mirrors app.py exactly):
    1. Look up the Yahoo Finance ticker for this portfolio ticker
    2. If the stock trades in pence (GBP_PENCE_TICKERS), divide by 100
    3. If the currency is not USD, multiply by the relevant FX rate
    4. Return None if any required data is missing

    Parameters
    ----------
    prices      : Full price DataFrame (equities + FX pairs)
    port_ticker : Internal portfolio ticker (key in TICKER_MAP)

    Returns
    -------
    pd.Series of USD prices, or None if data unavailable
    """
    yf_ticker = TICKER_MAP.get(port_ticker)
    if yf_ticker is None or yf_ticker not in prices.columns:
        return None

    # ── Start with local price ────────────────────────────────────────────────
    series = prices[yf_ticker].dropna().copy()

    # ── Pence → GBP correction ────────────────────────────────────────────────
    if yf_ticker in GBP_PENCE_TICKERS:
        series = series / 100.0

    # ── FX conversion to USD ──────────────────────────────────────────────────
    currency = CURRENCY_MAP.get(port_ticker, "USD")
    if currency != "USD":
        fx_col = FX_TICKERS.get(currency)
        if fx_col is None or fx_col not in prices.columns:
            return None  # FX data unavailable — skip this holding
        # Align FX to equity dates (some FX pairs trade different hours)
        fx_rate = prices[fx_col].reindex(series.index).ffill().bfill()
        series  = series * fx_rate

    return series


# ── Build full USD price matrix for all holdings ───────────────────────────────
usd_prices_dict = {
    row["Ticker"]: usd_price_series(prices, row["Ticker"])
    for _, row in holdings.iterrows()
}

# Keep only tickers where we successfully got a USD price series
available_tickers = [t for t, s in usd_prices_dict.items() if s is not None]
print(f"Tickers with USD prices: {len(available_tickers)} / {len(holdings)}")

# ── Assemble into a matrix, forward-fill, compute log returns ─────────────────
usd_px = (
    pd.DataFrame({t: usd_prices_dict[t] for t in available_tickers})
    .sort_index()
    .ffill()
)

usd_returns = (
    np.log(usd_px / usd_px.shift(1))
    .dropna()
)

# ── Normalise weights for available tickers only ───────────────────────────────
w = (
    holdings
    .set_index("Ticker")
    .loc[available_tickers, "w"]
    .pipe(lambda s: s / s.sum())   # re-normalise after dropping missing tickers
)

# ── Weighted portfolio return — dot product of returns matrix and weight vector ─
port_returns = usd_returns[available_tickers].dot(w)

# ── S&P 500 return series ──────────────────────────────────────────────────────
sp500_returns = (
    np.log(prices["^GSPC"] / prices["^GSPC"].shift(1))
    .dropna()
    .reindex(port_returns.index)
    .dropna()
)

port_returns = port_returns.reindex(sp500_returns.index).dropna()

print(f"\nPortfolio return series: {len(port_returns)} observations")
print(f"  Annualised return: {port_returns.mean() * TRADING_DAYS:.2%}")
print(f"  Annualised vol   : {port_returns.std() * np.sqrt(TRADING_DAYS):.2%}")
print(f"\nS&P 500 return series: {len(sp500_returns)} observations")
print(f"  Annualised return: {sp500_returns.mean() * TRADING_DAYS:.2%}")
print(f"  Annualised vol   : {sp500_returns.std() * np.sqrt(TRADING_DAYS):.2%}")


In [ ]:

# ── Cumulative return chart (replicates the app's Tab 1 performance chart) ─────

# exp(cumulative log-return) = compound growth factor
cum_port  = (port_returns.cumsum().apply(np.exp) * 100).rename("Portfolio")
cum_sp500 = (sp500_returns.cumsum().apply(np.exp) * 100).rename("S&P 500")

cum_df = (
    pd.concat([cum_port, cum_sp500], axis=1)
    .dropna()
)

total_port  = cum_df["Portfolio"].iloc[-1]  - 100
total_sp500 = cum_df["S&P 500"].iloc[-1] - 100

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(cum_df.index, cum_df["Portfolio"],
        lw=2.5, color="royalblue",
        label=f"Portfolio ({total_port:+.1f}%)")
ax.plot(cum_df.index, cum_df["S&P 500"],
        lw=2.5, color="crimson", ls="--",
        label=f"S&P 500 ({total_sp500:+.1f}%)")

ax.axhline(100, color="grey", lw=0.7, ls=":")
ax.set_title("Portfolio vs S&P 500 — Cumulative Return (USD, Base 100)",
             fontsize=14, fontweight="bold")
ax.set_ylabel("Growth of $100")
ax.set_xlabel("")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()


---
## Exercise 4 — CAPM Regression

### Background
The app runs an OLS regression of **monthly** portfolio returns on monthly S&P 500 returns:

$$R_{\text{portfolio},t} = \alpha + \beta \cdot R_{\text{S\&P500},t} + \epsilon_t$$

- $\alpha$ (alpha) = excess return not explained by market exposure (annualised: $(1+\alpha_{monthly})^{12}-1$)
- $\beta$ (beta) = sensitivity to the market (β > 1 = more volatile than market)
- $R^2$ = fraction of portfolio variance explained by the market

### Your Task
1. Convert daily returns to monthly (compound properly: $(1+r_{daily})^n - 1$)
2. Run the OLS regression using `scipy.stats.linregress`
3. Print a clean summary table
4. Plot the regression scatter with a fitted line using seaborn

> **Hint:** Use `.resample("ME")` to aggregate to month-end.


In [ ]:

# ── SOLUTION ──────────────────────────────────────────────────────────────────

def to_monthly_returns(daily_returns: pd.Series) -> pd.Series:
    """
    Compound daily log-returns to monthly simple returns.

    Uses resample to group by month-end, then compounds:
    R_monthly = prod(1 + r_daily) - 1
    """
    return (
        daily_returns
        .resample("ME")                              # group by calendar month-end
        .apply(lambda r: (1 + r).prod() - 1)        # compound daily → monthly
    )


# ── Monthly returns ────────────────────────────────────────────────────────────
port_monthly  = to_monthly_returns(port_returns)
sp500_monthly = to_monthly_returns(sp500_returns)

# Align on common months
common_months = port_monthly.index.intersection(sp500_monthly.index)
pm = port_monthly.loc[common_months]
sm = sp500_monthly.loc[common_months]

# ── OLS regression via scipy ───────────────────────────────────────────────────
slope, intercept, r_value, p_value, std_err = stats.linregress(sm.values, pm.values)

# Annualise alpha: (1 + monthly_alpha)^12 - 1
alpha_annualised = (1 + intercept) ** 12 - 1

# ── Print CAPM summary table (replicates app's capm-table) ────────────────────
capm_summary = pd.DataFrame([
    {"Metric": "Alpha (annualised)",    "Value": f"{alpha_annualised:+.2%}"},
    {"Metric": "Beta",                  "Value": f"{slope:.3f}"},
    {"Metric": "R²",                    "Value": f"{r_value**2:.3f}"},
    {"Metric": "Alpha p-value",         "Value": f"{p_value:.4f}"},
    {"Metric": "Monthly observations",  "Value": str(len(pm))},
    {"Metric": "Std error (slope)",     "Value": f"{std_err:.4f}"},
])

print("CAPM Regression — Monthly Returns vs S&P 500")
print("=" * 45)
print(capm_summary.to_string(index=False))
print()
if alpha_annualised > 0 and p_value < 0.05:
    print("✓  Statistically significant positive alpha — portfolio outperforms on risk-adjusted basis")
elif alpha_annualised > 0:
    print("△  Positive alpha but NOT statistically significant (p > 0.05)")
else:
    print("✗  Negative alpha — market exposure better explains returns")


In [ ]:

# ── Scatter plot with regression line ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

sns.regplot(
    x=sm.values * 100,
    y=pm.values * 100,
    ax=ax,
    scatter_kws={"alpha": 0.6, "s": 40, "color": "steelblue"},
    line_kws={"color": "crimson", "lw": 2},
    ci=95,
)

# Reference lines
ax.axhline(0, color="grey", lw=0.7, ls=":")
ax.axvline(0, color="grey", lw=0.7, ls=":")

# Annotation box
textstr = (f"α = {alpha_annualised:+.2%} p.a.\n"
           f"β = {slope:.3f}\n"
           f"R² = {r_value**2:.3f}\n"
           f"p(α) = {p_value:.4f}")
ax.text(0.05, 0.95, textstr, transform=ax.transAxes, va="top", fontsize=10,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="grey", alpha=0.8))

ax.set_title("CAPM Regression — Monthly Portfolio Returns vs S&P 500",
             fontweight="bold", fontsize=12)
ax.set_xlabel("S&P 500 Monthly Return (%)")
ax.set_ylabel("Portfolio Monthly Return (%)")
plt.tight_layout()
plt.show()

print("\nKey insight: Beta > 1 means the portfolio amplifies market moves;")
print("Beta < 1 means it is more defensive.  Alpha measures skill beyond market exposure.")


---
## Exercise 5 — Rolling 6-Month Volatility (Tab 1, Chart 2)

### Background
The app plots rolling 6-month (126 trading days) annualised volatility for
both the portfolio and the S&P 500.  This reveals whether the portfolio has
been more or less volatile than the benchmark at each point in time.

### Your Task
1. Compute rolling 126-day std deviation for both series, annualised to `%`
2. Plot both on one chart using seaborn/matplotlib
3. Add a shaded band to highlight periods where portfolio vol > S&P 500 vol

> **Hint:** `.rolling(126).std() * sqrt(252) * 100`


In [ ]:

# ── SOLUTION ──────────────────────────────────────────────────────────────────

ROLL_WINDOW = 126  # 6 calendar months ≈ 126 trading days

# ── Rolling volatility using method chaining ───────────────────────────────────
rolling_vol = (
    pd.DataFrame({"Portfolio": port_returns, "S&P 500": sp500_returns})
    .rolling(window=ROLL_WINDOW)              # rolling window object
    .std()                                     # rolling standard deviation
    .mul(np.sqrt(TRADING_DAYS) * 100)         # annualise and convert to percent
    .dropna()
)

# ── Flag periods where portfolio is MORE volatile than S&P 500 ────────────────
vol_excess = rolling_vol["Portfolio"] - rolling_vol["S&P 500"]

fig, ax = plt.subplots(figsize=(13, 5))

# Shaded band: portfolio vol > S&P 500 vol
ax.fill_between(
    rolling_vol.index,
    rolling_vol["Portfolio"],
    rolling_vol["S&P 500"],
    where=(rolling_vol["Portfolio"] > rolling_vol["S&P 500"]),
    alpha=0.15, color="royalblue", label="Portfolio more volatile"
)
ax.fill_between(
    rolling_vol.index,
    rolling_vol["Portfolio"],
    rolling_vol["S&P 500"],
    where=(rolling_vol["Portfolio"] <= rolling_vol["S&P 500"]),
    alpha=0.15, color="crimson", label="S&P 500 more volatile"
)

# Lines
ax.plot(rolling_vol.index, rolling_vol["Portfolio"],
        lw=2, color="royalblue", label="Portfolio")
ax.plot(rolling_vol.index, rolling_vol["S&P 500"],
        lw=2, color="crimson", ls="--", label="S&P 500")

ax.set_title(f"Rolling {ROLL_WINDOW}-Day Annualised Volatility — Portfolio vs S&P 500",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Annualised Volatility (%)")
ax.legend()
plt.tight_layout()
plt.show()

# ── Summary ────────────────────────────────────────────────────────────────────
pct_higher_vol = (rolling_vol["Portfolio"] > rolling_vol["S&P 500"]).mean()
print(f"Fraction of days portfolio vol > S&P 500 vol: {pct_higher_vol:.1%}")
print(f"Mean portfolio vol  : {rolling_vol['Portfolio'].mean():.1f}%")
print(f"Mean S&P 500 vol    : {rolling_vol['S&P 500'].mean():.1f}%")


---
## Exercise 6 — FX Return Decomposition (Tab 2)

### Background
The app decomposes each holding's USD return into:
1. **Local return** — how much the stock moved in its home currency
2. **FX impact** — how much the exchange rate moved (approximately: USD return − local return)

This is the core of **currency attribution** — a standard tool in international portfolio management.

### Your Task
1. For each holding, compute `local_return`, `usd_return`, and `fx_impact` over the full period
2. Aggregate to region level using weighted averages (weight = portfolio weight)
3. Produce the horizontal bar charts that appear in Tab 2

> **Note:** FX impact here is a **simple approximation**:
> $\text{FX impact} \approx R_{USD} - R_{local}$
> The exact decomposition is: $(1 + R_{USD}) = (1 + R_{local})(1 + R_{FX})$
> so $R_{FX} = \frac{1+R_{USD}}{1+R_{local}} - 1$, but the approximation is fine for small returns.


In [ ]:

# ── SOLUTION ──────────────────────────────────────────────────────────────────

def compute_holding_returns(prices: pd.DataFrame, holdings_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each holding compute total-period local return, USD return, and FX impact.

    Returns a DataFrame with one row per holding.
    """
    rows = []
    for _, row in holdings_df.iterrows():
        pticker  = row["Ticker"]
        yf_tick  = TICKER_MAP.get(pticker)
        currency = CURRENCY_MAP.get(pticker, "USD")

        if yf_tick is None or yf_tick not in prices.columns:
            continue

        # ── Local price series ─────────────────────────────────────────────────
        local_s = prices[yf_tick].dropna().copy()
        if yf_tick in GBP_PENCE_TICKERS:
            local_s = local_s / 100.0  # pence → GBP
        if local_s.empty:
            continue

        # ── Local return (start → end of window) ──────────────────────────────
        p0_local  = local_s.iloc[0]
        p1_local  = local_s.iloc[-1]
        local_ret = (p1_local / p0_local) - 1 if p0_local != 0 else np.nan

        # ── USD return ────────────────────────────────────────────────────────
        if currency == "USD":
            usd_ret = local_ret
            fx_ret  = 0.0
        else:
            fx_col = FX_TICKERS.get(currency)
            if fx_col is None or fx_col not in prices.columns:
                usd_ret = np.nan
                fx_ret  = np.nan
            else:
                # Convert local price to USD
                fx_s    = prices[fx_col].reindex(local_s.index).ffill().bfill()
                usd_s   = local_s * fx_s
                p0_usd  = usd_s.iloc[0]
                p1_usd  = usd_s.iloc[-1]
                usd_ret = (p1_usd / p0_usd) - 1 if p0_usd != 0 else np.nan
                fx_ret  = usd_ret - local_ret   # approximation

        rows.append({
            "Ticker":       pticker,
            "Name":         row["name"],
            "Weight (%)":   round(row["weight"], 2),
            "w":            row["w"],
            "Currency":     currency,
            "Region":       REGION_LABEL.get(currency, currency),
            "Local Return": local_ret,
            "USD Return":   usd_ret,
            "FX Impact":    fx_ret,
        })

    return pd.DataFrame(rows)


detail_df = compute_holding_returns(prices, holdings)

# ── Weighted average by region ─────────────────────────────────────────────────
def weighted_avg(group, col):
    """Weighted average of col using portfolio weights, skipping NaN."""
    valid = group.dropna(subset=[col])
    if valid.empty:
        return np.nan
    return np.average(valid[col], weights=valid["w"])

region_summary = (
    detail_df
    .groupby(["Currency", "Region"])
    .apply(lambda g: pd.Series({
        "Local Return (%)": weighted_avg(g, "Local Return") * 100,
        "FX Impact (%)":    weighted_avg(g, "FX Impact")    * 100,
        "USD Return (%)":   weighted_avg(g, "USD Return")   * 100,
    }), include_groups=False)
    .reset_index()
    .sort_values("USD Return (%)", ascending=False)
    .round(1)
)

print("Regional Return Decomposition:")
print(region_summary.to_string(index=False))


In [ ]:

# ── Replicate the app's Tab 2 charts ──────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left: Local currency returns (horizontal bar) ─────────────────────────────
colors_local = ["royalblue" if v >= 0 else "crimson"
                for v in region_summary["Local Return (%)"]]

axes[0].barh(region_summary["Region"], region_summary["Local Return (%)"],
             color=colors_local, alpha=0.8, edgecolor="white")
axes[0].axvline(0, color="grey", lw=0.8)

# Annotate bars
for i, (val, reg) in enumerate(zip(region_summary["Local Return (%)"],
                                    region_summary["Region"])):
    axes[0].text(val + (1 if val >= 0 else -1), i,
                 f"{val:+.1f}%", va="center", ha="left" if val >= 0 else "right",
                 fontsize=9)

axes[0].set_title("Weighted Return by Currency\n(Local Currency Terms)",
                  fontweight="bold")
axes[0].set_xlabel("Return (%)")
axes[0].invert_yaxis()

# ── Right: USD decomposition (stacked horizontal bar) ─────────────────────────
bar_width = 0.5
y_pos = range(len(region_summary))

axes[1].barh(y_pos, region_summary["Local Return (%)"],
             color="steelblue", alpha=0.8, label="Local Return", height=bar_width)
axes[1].barh(y_pos, region_summary["FX Impact (%)"],
             left=region_summary["Local Return (%)"],
             color="darkorange", alpha=0.8, label="FX Impact", height=bar_width)

axes[1].axvline(0, color="grey", lw=0.8)
axes[1].set_yticks(list(y_pos))
axes[1].set_yticklabels(region_summary["Region"])
axes[1].set_title("Return Decomposition\nLocal Return + FX Impact = USD Return",
                  fontweight="bold")
axes[1].set_xlabel("Return (%)")
axes[1].invert_yaxis()
axes[1].legend(loc="lower right")

plt.suptitle("Tab 2 FX & Regional Analysis — Notebook Replica",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("\nKey insight: FX can add to OR subtract from returns.")
print("A UK stock that rose 10% in GBP but where GBP/USD fell 5%")
print("would show: Local=+10%, FX=-5%, USD=+5%")


---
## Exercise 7 — Extension: Rolling Correlation Between Portfolio and S&P 500

### Background
The app doesn't currently show rolling correlation — but this is a natural addition
that would fit perfectly in Tab 1.  Rolling correlation shows whether the portfolio
has become more or less correlated with the market over time.

When correlation is high (→1), the portfolio tracks the index closely and diversification
benefits are minimal.  When correlation is low, the manager is adding genuine active risk.

### Your Task
1. Compute rolling 126-day correlation between portfolio and S&P 500 returns
2. Overlay rolling beta (slope of the regression in each window) on the same chart
3. Highlight when rolling beta exceeds 1.0 (more sensitive than the market)

> **Hint:** Use `.rolling(126).corr(other_series)` for correlation and
> `.rolling(126).cov(other) / .rolling(126).var()` for rolling beta.


In [ ]:

# ── SOLUTION ──────────────────────────────────────────────────────────────────

ROLL_W = 126

# ── Rolling correlation ────────────────────────────────────────────────────────
roll_corr = (
    port_returns
    .rolling(ROLL_W)
    .corr(sp500_returns)
    .dropna()
    .rename("Rolling Correlation")
)

# ── Rolling beta = Cov(portfolio, market) / Var(market) ───────────────────────
roll_beta = (
    port_returns
    .rolling(ROLL_W)
    .cov(sp500_returns)
    .div(sp500_returns.rolling(ROLL_W).var())
    .dropna()
    .rename("Rolling Beta")
)

# ── Align on common index ──────────────────────────────────────────────────────
roll_df = pd.concat([roll_corr, roll_beta], axis=1).dropna()

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# ── Top: Rolling correlation ───────────────────────────────────────────────────
axes[0].plot(roll_df.index, roll_df["Rolling Correlation"],
             lw=2, color="steelblue", label="Rolling Correlation (126d)")
axes[0].axhline(roll_df["Rolling Correlation"].mean(), color="grey", lw=1,
                ls="--", label=f"Mean = {roll_df['Rolling Correlation'].mean():.2f}")
axes[0].fill_between(roll_df.index, roll_df["Rolling Correlation"],
                     roll_df["Rolling Correlation"].mean(),
                     alpha=0.15, color="steelblue")
axes[0].set_ylabel("Correlation")
axes[0].set_ylim(-0.1, 1.1)
axes[0].set_title("Rolling Correlation — Portfolio vs S&P 500", fontweight="bold")
axes[0].legend()

# ── Bottom: Rolling beta ───────────────────────────────────────────────────────
axes[1].plot(roll_df.index, roll_df["Rolling Beta"],
             lw=2, color="darkorange", label="Rolling Beta (126d)")
axes[1].axhline(1.0, color="crimson", lw=1.2, ls="--", label="Beta = 1.0 (market)")
axes[1].axhline(roll_df["Rolling Beta"].mean(), color="grey", lw=1,
                ls=":", label=f"Mean beta = {roll_df['Rolling Beta'].mean():.2f}")

# Shade periods of high beta (>1)
axes[1].fill_between(roll_df.index, roll_df["Rolling Beta"], 1.0,
                     where=(roll_df["Rolling Beta"] > 1.0),
                     alpha=0.2, color="crimson", label="Beta > 1 (more volatile than market)")

axes[1].set_ylabel("Beta")
axes[1].set_title("Rolling Beta — Portfolio vs S&P 500", fontweight="bold")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

pct_beta_above1 = (roll_df["Rolling Beta"] > 1.0).mean()
print(f"Fraction of window where beta > 1: {pct_beta_above1:.1%}")
print("\nThis chart would be a valuable addition to the app's Tab 1.")
print("It reveals whether the portfolio's market sensitivity is stable or time-varying.")


---
## 🏁 Exercise Summary

Across these 7 exercises you have built the **complete analytical engine** of the
portfolio viewer app from scratch — entirely in pandas using method chaining.

| Exercise | What you built | App equivalent |
|----------|---------------|----------------|
| 1 | `download_all_prices()` | `download_prices()` |
| 2 | Holdings table with normalised weights | `holdings` construction |
| 3 | USD price conversion + portfolio returns | `usd_price_series()` + `build_tab1()` returns |
| 4 | CAPM regression + scatter plot | CAPM table in Tab 1 |
| 5 | Rolling 6-month volatility | Rolling vol chart in Tab 1 |
| 6 | FX return decomposition + regional charts | `build_tab2()` charts |
| 7 | Rolling correlation + rolling beta | Extension — not yet in app |

### What the app adds on top
The Dash app wraps this same logic in:
- An **interactive date slider** — filters `prices.loc[start:end]` before passing to `build_tab1/2`
- A **Refresh Prices button** — calls `download_prices()` and refreshes the slider range
- **Plotly** charts instead of matplotlib (hover tooltips, rangeslider, zoom)
- **DataTable** components with column sorting, filtering, and conditional formatting

The notebook is the best place to **develop and debug** the analytics.
The app is the best place to **present and explore** them interactively.
